In [1]:
import docx2txt
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path='./docs/소득세법_20260701.docx'

text = docx2txt.process(file_path)

# Document 생성
document = Document(
    page_content=text,
    metadata={'source': file_path}
)

# print(document.page_content[:500])

# 텍스트 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents([document])

# 결과 확인
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")

/home/dmin/miniconda3/envs/apiedu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


총 313개의 청크로 분할되었습니다.


In [2]:
print(f"첫 번째 청크 내용:\n{chunks[0].page_content}")

첫 번째 청크 내용:
소득세법



소득세법

[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]

재정경제부(재산세제과(양도소득세)) 044-215-4312

재정경제부(소득세제과(근로소득)) 044-215-4216

재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233

재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217

재정경제부(국제조세제도과(비거주자)) 044-215-4651



제1장 총칙 <개정 2009. 12. 31.>



제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.

[본조신설 2009. 12. 31.]

[종전 제1조는 제2조로 이동 <2009. 12. 31.>]



제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>

1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.

2. “비거주자”란 거주자가 아닌 개인을 말한다.

3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.

4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.

5. “사업자”란 사업소득이 있는 거주자를 말한다.

② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.

[본조신설 2009. 12. 31.]



제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다.

1. 거주자

2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인

② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다.

1. 거주자

2. 비거주자

3. 내국법인


In [3]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from dotenv import load_dotenv
import os

/tmp/ipykernel_52221/2533300891.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader


In [4]:
# 1. 환경 변수 로드
load_dotenv()

loader = Docx2txtLoader("./docs/소득세법_20260701.docx")

In [5]:
# 2. 문서 로드 및 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['제', '\n', '\n\n', ' ', '']
)

# documents = loader.load()
# chunks = text_splitter.split_documents(documents)

chunks = loader.load_and_split(text_splitter=text_splitter)

In [6]:
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")

총 311개의 청크로 분할되었습니다.


In [7]:
# 3. 문서 임베딩
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

database = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="my-tax-index"
)

In [8]:
# 4. Recursive 생성
recursive = database.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [16]:
# 5. Prompt 생성 => 객체 형태로 생성
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    '''
    [Identity]
    당신은 한국의 소득세법 전문가입니다.
    [Context]
    제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해주세요
    문서에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    마지막에는 반드시 "출처"를 명시해 주세요.
    
    [Context]
    {context}

    [Question]
    {query}
    
    '''    
)

In [39]:
# 6. LLM 생성
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model='gpt-4',
    model_provider='openai',
    temperature=0,
    max_tokens=1000
)

In [40]:
# 7. 문자열 출력 형식
def format_response(docs):
    return "\n\n".join([f"출처: {doc.metadata['source']}\n내요이 {doc.page_content}" for doc in docs])


In [41]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": recursive | format_response,
     "query": RunnablePassthrough()}
    |prompt_template
    |llm
    |StrOutputParser()
)

In [42]:
query = '종합과세 표준에 대해 설명해주세요'
result = rag_chain.invoke(query)

print(result)


    종합소득과세표준은 제16조, 제17조, 제19조, 제20조, 제20조의3, 제21조, 제24조부터 제26조까지, 제27조부터 제29조까지, 제31조부터 제35조까지, 제37조, 제39조, 제41조부터 제46조까지, 제46조의2, 제47조 및 제47조의2에 따라 계산한 이자소득금액, 배당소득금액, 사업소득금액, 근로소득금액, 연금소득금액 및 기타소득금액의 합계액(이하 “종합소득금액”이라 한다)에서 제50조, 제51조, 제51조의3, 제51조의4 및 제52조에 따른 공제(이하 “종합소득공제”라 한다)를 적용한 금액을 말합니다.

그러나, 다음 각 호에 따른 소득의 금액은 종합소득과세표준을 계산할 때 합산하지 않습니다.
1. 「조세특례제한법」 또는 이 법 제12조에 따라 과세되지 아니하는 소득
2. 대통령령으로 정하는 일용근로자의 근로소득
3. 제129조제2항의 세율에 따라 원천징수하는 이자소득 및 배당소득과 제16조제1항제10호에 따른 직장공제회 초과반환금
4. 법인으로 보는 단체 외의 단체 중 수익을 구성원에게 배분하지 아니하는 단체로서 단체명을 표기하여 금융거래를 하는 단체가 「금융실명거래 및 비밀보장에 관한 법률」 제2조제1호 각 목의 어느 하나에 해당하는 금융회사등으로부터 받는 이자소득 및 배당소득
5. 「조세특례제한법」에 따라 분리과세되는 소득
6. 제3호부터 제5호까지의 규정 외의 이자소득과 배당소득(제17조제1항제8호에 따른 배당소득은 제외한다)으로서 그 소득의 합계액이 2천만원 이하인 경우

출처: ./docs/소득세법_20260701.docx
